# Проверка гипотезы по ФП группы 28 (МкБ/МСБ)

Гипотеза: факторы группы `28*` обычно возникают после факторов групп `14*`, `15*`, `16*`, `24*`, `26*`.

Когорты:
1. ИНН, у которых **первый ФП** (в выборке) возник в 2024 году.
2. ИНН, у которых **первый ФП группы 28** (в выборке) возник в 2024 году.

База готовится по логике `analysis_crm_segments.ipynb` (1-в-1), затем ограничивается сегментами `МкБ`/`МСБ` и типом `ФП`.

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)

# ── Пути к файлам ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"
REF_FILE = f"{DATA_DIR}/ref_book.csv"

# ── Период анализа ──
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

TARGET_SEGMENTS = ["МкБ", "МСБ"]
FP_TYPE = "ФП"
PRIOR_GROUPS = ["14", "15", "16", "24", "26"]
TARGET_GROUP = "28"


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def starts_with_group(code: str, group_prefix: str) -> bool:
    code = str(code).strip()
    return code.startswith(group_prefix)


def detect_prior_group(code: str):
    code = str(code).strip()
    for g in PRIOR_GROUPS:
        if code.startswith(g):
            return g
    return None

print("Параметры заданы.")
print(f"Период: {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}")

In [ ]:
# ── Подготовка данных: 1-в-1 с analysis_crm_segments.ipynb ──
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

df["inn"] = df["X_INN"].apply(normalize_inn)

df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

if "VAL" in df.columns:
    before_val = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): {len(df):,} строк (удалено {before_val - len(df):,})")

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})

df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print("\nНемаппированные значения X_AREA_RESP (будут исключены):")
    print(unmapped.to_string())

df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + fp_num + TYPE_FP + DATE): {len(df):,} строк (удалено {before_dedup - len(df):,} дублей)")

# Ограничение задачи: только МкБ/МСБ и только ФП.
df_fp = df[(df["segment"].isin(TARGET_SEGMENTS)) & (df["TYPE_FP"] == FP_TYPE)].copy()
print("\nПосле ограничения на МкБ/МСБ + TYPE_FP='ФП':")
print(f"  Строк: {len(df_fp):,}")
print(f"  Уникальных ИНН: {df_fp['inn'].nunique():,}")
print(df_fp.groupby('segment')['inn'].nunique().rename('inn_nunique').to_string())

In [ ]:
# ── Построение когорт ──
# Когорта 1: первый ФП (в выборке) в 2024.
first_fp_by_inn_seg = (
    df_fp.groupby(["segment", "inn"], as_index=False)
    .agg(first_fp_dt=("dt", "min"))
)
cohort1 = first_fp_by_inn_seg[first_fp_by_inn_seg["first_fp_dt"].dt.year == 2024][["segment", "inn"]].copy()
cohort1["cohort"] = "cohort1_first_fp_2024"

# Когорта 2: первый 28* ФП (в выборке) в 2024.
first_28_by_inn_seg = (
    df_fp[df_fp["fp_num"].astype(str).str.startswith(TARGET_GROUP)]
    .groupby(["segment", "inn"], as_index=False)
    .agg(first_28_dt=("dt", "min"))
)
cohort2 = first_28_by_inn_seg[first_28_by_inn_seg["first_28_dt"].dt.year == 2024][["segment", "inn"]].copy()
cohort2["cohort"] = "cohort2_first_28_2024"

print("Размеры когорт (ИНН):")
print(
    pd.concat([
        cohort1.groupby("segment")["inn"].nunique().rename("cohort1_inn"),
        cohort2.groupby("segment")["inn"].nunique().rename("cohort2_inn"),
    ], axis=1).fillna(0).astype(int).to_string()
)

In [ ]:
def compute_metrics_for_cohort(cohort_df: pd.DataFrame, cohort_name: str):
    # База когорты
    c = cohort_df[["segment", "inn"]].drop_duplicates().copy()
    c["cohort"] = cohort_name

    cohort_events = df_fp.merge(c[["segment", "inn"]], on=["segment", "inn"], how="inner")

    # Первая дата 28* внутри когорты (может отсутствовать для части ИНН)
    first_28 = (
        cohort_events[cohort_events["fp_num"].astype(str).str.startswith(TARGET_GROUP)]
        .groupby(["segment", "inn"], as_index=False)
        .agg(first_28_dt=("dt", "min"))
    )

    # События 28* в когорте
    events_28 = (
        cohort_events[cohort_events["fp_num"].astype(str).str.startswith(TARGET_GROUP)]
        .groupby("segment", as_index=False)
        .agg(events_28=("fp_num", "size"), inn_with_28=("inn", "nunique"))
    )

    # События до first_28_dt (только для ИНН, у которых 28* есть)
    before_28 = cohort_events.merge(first_28, on=["segment", "inn"], how="inner")
    before_28 = before_28[before_28["dt"] < before_28["first_28_dt"]].copy()
    before_28["prior_group"] = before_28["fp_num"].apply(detect_prior_group)
    before_28 = before_28[before_28["prior_group"].notna()].copy()

    by_group = (
        before_28.groupby(["segment", "prior_group"], as_index=False)
        .agg(
            prior_events_before_28=("fp_num", "size"),
            inn_with_group_before_28=("inn", "nunique"),
        )
    )

    # Подготовка denominator для долей
    inn_with_28_by_seg = events_28[["segment", "inn_with_28"]].copy()

    # Доли по каждой группе
    by_group = by_group.merge(inn_with_28_by_seg, on="segment", how="left")
    by_group["share_inn_with_group_before_28"] = np.where(
        by_group["inn_with_28"] > 0,
        by_group["inn_with_group_before_28"] / by_group["inn_with_28"],
        np.nan,
    )
    by_group.insert(0, "cohort", cohort_name)

    # Ключевая доля any_prior
    any_prior = (
        before_28[["segment", "inn"]]
        .drop_duplicates()
        .groupby("segment", as_index=False)
        .agg(inn_with_any_prior_before_28=("inn", "nunique"))
    )

    cohort_size = (
        c.groupby("segment", as_index=False)
        .agg(cohort_inn=("inn", "nunique"))
    )

    summary = cohort_size.merge(events_28, on="segment", how="left")
    summary = summary.merge(any_prior, on="segment", how="left")

    # Заполняем нулями
    for col in ["events_28", "inn_with_28", "inn_with_any_prior_before_28"]:
        summary[col] = summary[col].fillna(0).astype(int)

    summary["share_inn_with_28"] = np.where(summary["cohort_inn"] > 0, summary["inn_with_28"] / summary["cohort_inn"], np.nan)
    summary["share_inn_any_prior"] = np.where(summary["inn_with_28"] > 0, summary["inn_with_any_prior_before_28"] / summary["inn_with_28"], np.nan)

    # Добавляем доли по каждой группе в summary
    group_shares = (
        by_group.pivot_table(
            index="segment",
            columns="prior_group",
            values="share_inn_with_group_before_28",
            aggfunc="max",
        )
        .reset_index()
        .rename(columns={
            "14": "share_inn_prior_14",
            "15": "share_inn_prior_15",
            "16": "share_inn_prior_16",
            "24": "share_inn_prior_24",
            "26": "share_inn_prior_26",
        })
    )

    summary = summary.merge(group_shares, on="segment", how="left")
    summary.insert(0, "cohort", cohort_name)

    # Для полной видимости добавим нулевые строки по группам, если в сегменте нет событий
    seg_group_base = pd.MultiIndex.from_product([TARGET_SEGMENTS, PRIOR_GROUPS], names=["segment", "prior_group"]).to_frame(index=False)
    by_group = seg_group_base.merge(by_group, on=["segment", "prior_group"], how="left")
    by_group["cohort"] = by_group["cohort"].fillna(cohort_name)
    for col in ["prior_events_before_28", "inn_with_group_before_28", "inn_with_28"]:
        by_group[col] = by_group[col].fillna(0).astype(int)
    by_group["share_inn_with_group_before_28"] = np.where(
        by_group["inn_with_28"] > 0,
        by_group["inn_with_group_before_28"] / by_group["inn_with_28"],
        np.nan,
    )

    return summary, by_group


summary1, by_group1 = compute_metrics_for_cohort(cohort1, "cohort1_first_fp_2024")
summary2, by_group2 = compute_metrics_for_cohort(cohort2, "cohort2_first_28_2024")

cohort_segment_summary = pd.concat([summary1, summary2], ignore_index=True)
before_28_by_group = pd.concat([by_group1, by_group2], ignore_index=True)

# Формат порядка вывода
cohort_order = ["cohort1_first_fp_2024", "cohort2_first_28_2024"]
segment_order = ["МкБ", "МСБ"]

cohort_segment_summary["cohort"] = pd.Categorical(cohort_segment_summary["cohort"], categories=cohort_order, ordered=True)
cohort_segment_summary["segment"] = pd.Categorical(cohort_segment_summary["segment"], categories=segment_order, ordered=True)
cohort_segment_summary = cohort_segment_summary.sort_values(["cohort", "segment"]).reset_index(drop=True)

before_28_by_group["cohort"] = pd.Categorical(before_28_by_group["cohort"], categories=cohort_order, ordered=True)
before_28_by_group["segment"] = pd.Categorical(before_28_by_group["segment"], categories=segment_order, ordered=True)
before_28_by_group = before_28_by_group.sort_values(["cohort", "segment", "prior_group"]).reset_index(drop=True)

print("cohort_segment_summary")
display(cohort_segment_summary)
print("\nbefore_28_by_group")
display(before_28_by_group)

In [ ]:
# Детализация по ИНН (для ручной верификации при необходимости)
first_28_all = (
    df_fp[df_fp["fp_num"].astype(str).str.startswith(TARGET_GROUP)]
    .groupby(["segment", "inn"], as_index=False)
    .agg(first_28_dt=("dt", "min"))
)

prior_detail = df_fp.merge(first_28_all, on=["segment", "inn"], how="inner")
prior_detail = prior_detail[prior_detail["dt"] < prior_detail["first_28_dt"]].copy()
prior_detail["prior_group"] = prior_detail["fp_num"].apply(detect_prior_group)
prior_detail = prior_detail[prior_detail["prior_group"].notna()].copy()

print("Проверочная детализация (первые 50 строк):")
display(prior_detail.sort_values(["segment", "inn", "dt"]).head(50))

In [ ]:
OUT_XLSX = Path(DATA_DIR) / "fp28_hypothesis_cohorts.xlsx"

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    cohort_segment_summary.to_excel(writer, sheet_name="cohort_segment_summary", index=False)
    before_28_by_group.to_excel(writer, sheet_name="before_28_by_group", index=False)

print(f"Сохранено: {OUT_XLSX}")